# Python Best Practices & OOP: post-training exercises

In these exercises we will work with the building archetypes dataset. The questions draw on content from both *Python Best Practices* and *Object-Oriented Programming with Python*.

The questions come in two groups:

- **Q1-Q7** work through the material a step at a time. Everything you need is in your course notebooks.
- **Q8-Q10** are marked *Harder* and build on your answers to Q1-Q7.

The model solutions can be found in the `solutions.ipynb` notebook. We also have:

- `bad_function.py` - use for a code styling exercise below.

## The data

Sitting alongside this notebook is a single file, `buildings.csv`. Each row is one archetype (one kind of home in one place), and the columns you'll use are:

| Column | Contents |
|---|---|
| `region` | One of nine UK regions |
| `property_type` | `Detached`, `Flat`, `Bungalow`, ... |
| `heating_system` | `Gas boiler`, `ASHP`, `Oil boiler`, ... |
| `floor_area` | Floor area in m² |
| `stock` | Number of real dwellings this archetype stands for |
| `space_heating_co2` | Heating emissions in kgCO2 per year |

Run the cell below to load pandas, which we will use for many of the questions.

In [ ]:
# Run this cell to load the library we'll need
import pandas as pd

## Q1) Get to know the data

### Q1a)

Read `buildings.csv` into a `DataFrame` called `buildings`, and preview it with `.head()`.

### Q1b)

How many rows and columns does it have? Use `.shape`.

### Q1c)

What heating systems appear in the data? Use `.unique()` on the `heating_system` column.

### Q1d)

How many archetypes use a `"Gas boiler"`? Build a boolean mask and count the `True` values with `.sum()`.

In [ ]:
# Your Q1a) - Q1d) code here

## Q2) A function with a clear contract

Reading and filtering the data is something you'll do again and again, so it's a natural analysis step for our first function.

### Q2a)

Write a function `load_buildings()` that reads `buildings.csv` and returns the `DataFrame`. Give the path a **default argument** (`path="buildings.csv"`) so the function reads that file unless told otherwise. This is what makes the function general: the same code can load a different file later without being rewritten.

### Q2b)

Add a second argument, `region`, that defaults to `None`. When a region is given, the function should return only that region's rows; when it isn't, it should return them all.

*Hint*: `if region is not None:` is the test you want.

### Q2c)

Give the function a docstring saying what it does, what its arguments are and what it returns.

### Q2d)

Call `load_buildings(region="South England")` and check with `.shape[0]` that you got fewer rows than the full dataset.

Then, in a comment, answer this: your function reads a file from disk every time it's called. Is it a *pure* function - one whose output depends only on its inputs? What does that mean for testing it?

In [ ]:
# Your Q2a) - Q2d) code here

## Q3) Helpful errors

Here's a rough carbon-intensity lookup: illustrative kgCO2 for each kWh of fuel burned.

Run the cell below to define it, then use it in your answers.


In [ ]:
# Illustrative carbon factors - kgCO2 per kWh of fuel
CARBON_FACTORS = {
    "Gas boiler": 0.183,
    "Oil boiler": 0.246,
    "Biofuel boiler": 0.020,
    "ASHP": 0.000,
}

### Q3a)

Write a function `heating_emissions(fuel_kwh, system="Gas boiler")` that looks `system` up in `CARBON_FACTORS` and returns `fuel_kwh` multiplied by that factor.

### Q3b)

Call it with a system that isn't in the dictionary, for example `heating_emissions(10000, "Coal")`. What error do you get, and would it help a user who hasn't seen the source code?

### Q3c)

Rewrite the function with a `try` / `except` so that:

- an unknown `system` raises a `ValueError` whose message lists the systems that *are* allowed, and
- a non-numeric `fuel_kwh` (say the string `"lots"`) raises a `TypeError` with a clear message.

Test both. This is the same `KeyError` -> `ValueError` pattern you saw with `convert_age()` in Python Best Practices Chapter 2.

In [ ]:
# Your Q3a) - Q3c) code here

## Q4) Code style and naming

The snippet below works, but it reads badly. Examine it and try to understand what it does.


In [ ]:
import pandas as pd
def MeanInt(D):
 x=D['space_heating_co2']/D['floor_area']
 return x.mean()
DF=pd.read_csv('buildings.csv')
print( MeanInt(DF) )

### Q4a)

The function is supposed to compute the mean "intensity" (i.e., the mean CO2 space heating emissions per floor area).

Rewrite it to follow the conventions from the course:

- a descriptive function name in `snake_case`, and an argument name a reader can understand,
- a docstring,
- consistent quotes and PEP8 spacing (a blank line before the function, spaces around `=` and operators).

### Q4b)

You don't have to spot every style issue by eye. In your terminal, with the environment active, `ruff` will check and fix a lot of it for you:

```
ruff check .
ruff format .
```

Try running the ruff commands above to format the "bad_function.py" script and list some of the changes it makes (and *doesn't* make) below.

In [ ]:
# Your Q4a) code here

## Q5) Your first class

A single archetype has a name, a floor area, a heating system and an emissions figure that belong together - which is exactly what a class is for.

### Q5a)

Define a `Dwelling` class whose `__init__()` takes `name`, `floor_area`, `heating_system` and `co2`, and stores each as an attribute.

### Q5b)

Create an instance of your class from a real archetype - a detached, gas-heated home in South England:

```python
home = Dwelling("Detached, South England", 102.8, "Gas boiler", 872.01)
```

Check you can read `home.floor_area` and `home.heating_system` back out.

### Q5c)

Add a method `is_low_carbon(self)` that returns `True` when the dwelling's `co2` is `0`, and `False` otherwise. Test it on `home`.

*Hint*: `self.co2 == 0` gives you a `True` or `False` output, as we saw in the Programming with Python course.

In [ ]:
# Your Q5a) - Q5c) code here

## Q6) Methods that compute, and methods that change

Methods can return a value, and they can also change what the object stores.

### Q6a)

Add a method `intensity(self)` that returns the dwelling's emissions per square metre, `co2 / floor_area`.

### Q6b)

Retrofitting a home changes its heating system and its emissions. Add a method `retrofit(self, new_system, new_co2)` that overwrites `self.heating_system` and `self.co2` with the new values.

### Q6c)

You'll want to know whether a dwelling has been retrofitted. In `__init__()`, add an attribute `retrofitted` set to `False`, and have `retrofit()` set it to `True`. Declaring it in the constructor - even before it's needed - means every `Dwelling` has the attribute from the moment it's made.

### Q6d)

Test it: print `home.intensity()`, then `home.retrofit("ASHP", 0.0)`, then check that `home.is_low_carbon()` is now `True` and `home.retrofitted` is `True`.

During the course, you saw the convention of prefixing an internal attribute with `_`. Would `retrofitted` deserve one? Say why or why not in a comment.

In [ ]:
# Your Q6a) - Q6d) code here

## Q7) A class that owns its data

Classes are good at storing datasets together with the operations you expect the user to perform on them.

### Q7a)

Write a `BuildingsDataset` class whose `__init__()` takes a `filepath`, stores it, and sets `self.raw_data = None`. Setting it to `None` up front declares the attribute before the data has been read.

### Q7b)

Add a `read(self)` method that reads the csv at `self.filepath` into `self.raw_data`.

### Q7c)

Add a `mean_intensity(self)` method that returns the mean of `space_heating_co2 / floor_area` across `self.raw_data`.

### Q7d)

Give the class and each method a docstring. Then test it:

```python
ds = BuildingsDataset("buildings.csv")
print(ds.raw_data)      # None, before reading
ds.read()
print(ds.mean_intensity())
```

In [ ]:
# Your Q7a) - Q7d) code here

## Q8) Inheritance

*Harder*

A dwelling isn't the only thing that emits. A whole sector does too. They share some behaviours and properties and differ in others, which is exactly what inheritance is for.

### Q8a)

Write a class `EmissionsSource` whose `__init__()` takes `name`, `year` and `emissions`. Give it two methods:

- `cumulative_emissions(self, years)`, returning `self.emissions * years` - the emissions if this year's rate simply held for that many years.
- `describe(self)`, returning a sentence such as `"Aviation emitted 30.1 MtCO2e in 2025."`.

### Q8b)

Write a subclass `Dwelling(EmissionsSource)` that adds `floor_area` and `heating_system`. Use `super().__init__()` to get the shared attributes rather than repeating them. Then give `Dwelling` a method of its own, `intensity(self)`, returning `emissions / floor_area`. Only a dwelling has a floor area, so this method belongs on the subclass, not the parent.

### Q8c)

A dwelling can describe itself more fully than a bare emissions source can. **Override** `describe()` in Dwelling so it also names the heating system. Don't write the sentence out again. Instead, call `super().describe()` to get the parent's sentence, then add to it.

_Hint_: `super().describe()` runs the `EmissionsSource` version; you just append to whatever string it returns.

### Q8d)

Write a second subclass `Sector(EmissionsSource)` that adds nothing of its own, and does not override `describe()`.

### Q8e)

Put one instance of each subclass in a list and loop over them, calling `describe()` and `cumulative_emissions(5)` on every item. Use the instances below:

```python
scottish_bungalow = Dwelling("Scottish Bungalow", 2025, 4.32, 151.54, "Gas boiler")
aviation = Sector("Aviation", 2025, 30.1)
```

The same two calls do the right thing for each type: `Dwelling` uses its own `describe()`, `Sector` falls back to the inherited one, and both share `cumulative_emissions()`. In a comment, say what happens if you call `aviation.intensity()`, and why.

In [ ]:
# Your Q8a) - Q8e) code here

## Q9) Tests

*Harder*

You now have functions and classes which could be moved into a .py script or a python package. Before we do that, it's worth writing down some unit tests. Not only will it make the code more trustworthy, but it can also help to pin down what "correct" means before you start to rely on the code.

### Q9a)

In the cell below, write a function `test_heating_emissions()` that checks `heating_emissions(10000, "Gas boiler")` comes out at `1830.0` (to a sensible number of decimals using the `round()` function). A test is just a function that raises `AssertionError` when the code is wrong, so an `assert` statement is all you need.

### Q9b)

Write `test_unknown_system_raises()` that checks an unknown system raises `ValueError`. `pytest` gives you a clean way to assert that an error is raised:

```python
import pytest

def test_unknown_system_raises():
    with pytest.raises(ValueError):
        heating_emissions(10000, "Coal")
```

### Q9c)

Move both tests into a file called `test_buildings.py`, next to this notebook, along with the `heating_emissions` function and `CARBON_FACTORS` they rely on. From the terminal, with the environment active, run:

```
pytest
```

`pytest` finds any file named `test_*.py` and runs any function named `test_*` inside it. In a comment, note how many tests it collected and whether they passed.

In [ ]:
# Your Q9a) - Q9b) code here

## Q10) Into a module, and reproducible

*Harder*

Notebooks are for exploring. Once code is worth reusing, it belongs in a script you can import.

### Q10a)

Create a file called `buildings_tools.py` next to this notebook. Move `load_buildings()` (Q2), `heating_emissions()` with its `CARBON_FACTORS` (Q3), and your `Dwelling` and `EmissionsSource` classes into it. Import `pandas` at the top of the script.

### Q10b)

Look at what's now in the script. `load_buildings()`, `heating_emissions()` and `Dwelling` are all about buildings. `EmissionsSource` is not - a sector of the economy or a vehicle fleet can be an emissions source too. Generic code that other code might reuse is worth keeping apart from the buildings-specific code.

Create a second file, `helpers.py`, and move `EmissionsSource` into it. Then, at the top of `buildings_tools.py`, import it back:

```python
from helpers import EmissionsSource
```

`Dwelling` is still a subclass of `EmissionsSource`, so this import is what lets it find its parent.

### Q10c)

Import `load_buildings` and `Dwelling` back into this notebook with `from buildings_tools import load_buildings, Dwelling`, and check these still work.

_Hint_: you don't import `helpers` here - importing `Dwelling` quietly pulls in `EmissionsSource` behind the scenes, because the latter is imported by `buildings_tools.py`.

### Q10d)

Add a block at the end of the `buildings_tools.py`script, wrapped in `if __name__ == "__main__":`, that loads the data using `data = load_buildings()` and then computes the mean intensity with `intensity = (data["space_heating_co2"] / data["floor_area"]).mean()`.

From this notebook, run `%run buildings_tools.py` and, separately, `import buildings_tools`. Only one of them should print anything.

_Hint_: see Chapter 4 of the Programming with Python course for a refresher on `if __name__ == "__main__"` and its purpose.

In [ ]:
# Your Q10c) - Q10d) code here, once you've written buildings_tools.py and helpers.py